In [ ]:
import sys
import os

if not os.path.exists('pyolimp'):
    !git clone https://github.com/pyolimp/pyolimp.git

sys.path.insert(0, os.path.join(os.getcwd(), 'pyolimp'))

try:
    from olimp.dataset.olimp import olimp
    from olimp.dataset import read_img_path
    print("SUCCESS")
except ImportError as e:
    print(f"FAILED: {e}")
    

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import torchvision.models as models
import os
 
from olimp.dataset.olimp import olimp
from olimp.dataset import read_img_path

In [ ]:

from skimage.metrics import peak_signal_noise_ratio, structural_similarity


from olimp.dataset.olimp import olimp
from olimp.dataset import read_img_path

In [ ]:
# 2: Конфигурация (1000 вариантов, 1000 К , 1000 К sin)

PATCH_SIZE        = 256       # было 128 — теперь видим 3-6 периодов маски
SIGMA             = 10 / 255.0
TRAIN_RATIO       = 0.85      # 850 train / 150 val по изображениям 
COND_DIM          = 4         # размер вектора условия для FiLM


PATCHES_PER_EPOCH = 50_000    # on-the-fly патчей за эпоху (train)
VAL_PATCHES       = 5_000     # фиксированные патчи для val
 
BATCH_SIZE        = 32
NUM_WORKERS       = 4
EPOCHS            = 300
PATIENCE          = 30
LR                = 1e-3
 
CACHE_IMAGES      = True      # предзагрузить все изображения в RAM 
 
CHECKPOINT_PATH   = "checkpoint.pt"   # текущий чекпоинт (каждые N эпох)
BEST_MODEL_PATH   = "best_model.pt"   # лучшая модель по val_loss
CHECKPOINT_EVERY  = 5                 # сохранять чекпоинт каждые N эпох
 

In [ ]:
# 3: Загрузка датасета — 1000 изображений из разных категорий

CATEGORIES = [
    'abstracts and textures', 'abstracts and textures/abstract art', 'abstracts and textures/backgrounds and patterns', 'abstracts and textures/colorful abstracts', 'abstracts and textures/geometric shapes', 'abstracts and textures/neon abstracts', 'abstracts and textures/textures', 'animals', 'animals/birds', 'animals/farm animals', 'animals/insects and spiders', 'animals/marine life', 'animals/pets', 'animals/wild animals', 'art and culture', 'art and culture/cartoon and comics', 'art and culture/crafts and handicrafts', 'art and culture/dance and theater performances', 'art and culture/music concerts and instruments', 'art and culture/painting and frescoes', 'art and culture/sculpture and bas-reliefs', 'food and drinks', 'food and drinks/desserts and bakery', 'food and drinks/dishes', 'food and drinks/drinks', 'food and drinks/food products on store shelves', 'food and drinks/fruits and vegetables', 'food and drinks/street food', 'interiors', 'interiors/gyms and pools', 'interiors/living spaces', 'interiors/museums and galleries', 'interiors/offices', 'interiors/restaurants and cafes', 'interiors/shopping centers and stores', 'nature', 'nature/beaches', 'nature/deserts', 'nature/fields and meadows', 'nature/forest', 'nature/mountains', 'nature/water bodies', 'objects and items', 'objects and items/books and stationery', 'objects and items/clothing and accessories', 'objects and items/electronics and gadgets', 'objects and items/furniture and decor', 'objects and items/tools and equipment', 'objects and items/toys and games', 'portraits and people', 'portraits and people/athletes and dancers', 'portraits and people/crowds and demonstrations', 'portraits and people/group photos', 'portraits and people/individual portraits', 'portraits and people/models on runway', 'portraits and people/workers in their workplaces', 'sports and active leisure', 'sports and active leisure/cycling and rollerblading', 'sports and active leisure/extreme sports', 'sports and active leisure/individual sports', 'sports and active leisure/martial arts', 'sports and active leisure/team sports', 'sports and active leisure/tourism and hikes', 'text and pictogram', 'text and pictogram/billboard text', 'text and pictogram/blueprints', 'text and pictogram/caricatures and pencil drawing', 'text and pictogram/text documents', 'text and pictogram/traffic signs', 'urban scenes', 'urban scenes/architecture', 'urban scenes/city at night', 'urban scenes/graffiti and street art', 'urban scenes/parks and squares', 'urban scenes/streets and avenues', 'urban scenes/transport'
]
 
all_paths = []
for cat in CATEGORIES:
    try:
        ds    = olimp(categories={cat})
        paths = ds[cat]
        all_paths.extend(paths)
        
    except Exception as e:
        print(f"  {cat}: пропущено ({e})")
 
# Перемешать и взять 1000
rng = np.random.default_rng(42)
rng.shuffle(all_paths)
all_paths = all_paths[:1000]
print(f"\nИтого загружено: {len(all_paths)} изображений")
 
# Разбить по изображениям 
n_train     = int(TRAIN_RATIO * len(all_paths))
train_paths = all_paths[:n_train]
val_paths   = all_paths[n_train:]
print(f"Train: {len(train_paths)} изображений | Val: {len(val_paths)} изображений")

In [ ]:
# 4: Генерация 1000 конфигураций K + 1000 K sin

_cfg_rng = np.random.default_rng(0)
 
#  1000 горизонтальных полос (разная ширина: 4–128 px) 
stripe_h_configs = [
    {"type": "h", "width": float(w)}
    for w in _cfg_rng.uniform(4, 128, 1000)
]
 
#  1000 синусоидных масок (случайные freq, angle, phase) 
sin_configs = [
    {
        "type": "s",
        "freq":  float(f),
        "angle": float(a),
        "phase": float(p),
    }
    for f, a, p in zip(
        _cfg_rng.uniform(1 / 120, 1 / 8, 1000),   # период 8–120 px
        _cfg_rng.uniform(0, 180, 1000),             # любое направление
        _cfg_rng.uniform(0, 2 * np.pi, 1000),
    )
]
 
ALL_CONFIGS = stripe_h_configs + sin_configs
print(f"Всего конфигураций K: {len(ALL_CONFIGS)}  "
      f"(горизонтальных: {len(stripe_h_configs)}, синусоидальных: {len(sin_configs)})")
 
 
def make_mask(h: int, w: int, config: dict) -> np.ndarray:
    """
    Генерирует маску K ∈ [0,1] по конфигурации.
      type='h'  — бинарные горизонтальные полосы
      type='s'  — 2D синусоидальная маска sin(ax + by + phase)
    
    """
    if config["type"] == "h":
        sw = max(1, int(config["width"]))
        y  = np.arange(h)
        K  = ((y // sw) % 2).astype(np.float32)
        return K[:, None] * np.ones((1, w), dtype=np.float32)
 
    # type == 's'
    freq  = config["freq"]
    angle = np.radians(config["angle"])
    phase = config["phase"]
    a = 2 * np.pi * freq * np.cos(angle)
    b = 2 * np.pi * freq * np.sin(angle)
    xs = np.arange(w, dtype=np.float32)[None, :]   # (1, W)
    ys = np.arange(h, dtype=np.float32)[:, None]   # (H, 1)
    return ((np.sin(a * xs + b * ys + phase) + 1) / 2).astype(np.float32)
 
 
def config_to_cond(config: dict) -> np.ndarray:
    """
    4-мерный вектор условия для FiLM:
    [type_flag, normalized_period, normalized_angle, normalized_phase]
    """
    if config["type"] == "h":
        return np.array([
            0.0,
            (config["width"] - 4) / (128 - 4),   # ширина → [0, 1]
            0.0,
            0.0,
        ], dtype=np.float32)
    else:
        freq_min, freq_max = 1 / 120, 1 / 8
        return np.array([
            1.0,
            (config["freq"] - freq_min) / (freq_max - freq_min),
            config["angle"] / 180.0,
            config["phase"] / (2 * np.pi),
        ], dtype=np.float32)

In [ ]:
import subprocess
subprocess.run(["pip", "install", "numpy==1.26.4"], check=True)

In [ ]:
# 5: Вспомогательные функции

def load_rg(img_path):
    img = read_img_path(img_path)          # (C, H, W), uint8, CPU

    # np.array() работает с numpy 2.x в отличие от .numpy()
    img = np.array(img, dtype=np.float32) / 255.0   # (C, H, W)
    img = np.transpose(img, (1, 2, 0))               # (H, W, C)
    img = cv2.resize(img, (512, 512))                # (512, 512, C)

    # Grayscale → 3 канала
    if img.ndim == 2:
        img = np.stack([img, img, img], axis=-1)
    elif img.shape[2] == 1:
        img = np.concatenate([img, img, img], axis=-1)

    return img[:, :, :2].astype(np.float32)
 
def make_measurement(rg: np.ndarray, K: np.ndarray) -> np.ndarray:
    """
    P = K * R^0.7 + (1-K) * G^2 + N(0, σ)
 

    """
    R = rg[:, :, 0]
    G = rg[:, :, 1]
    P = K * R ** 0.7 + (1 - K) * G ** 2
    noise = np.random.normal(0, SIGMA, P.shape).astype(np.float32)
    return np.clip(P + noise, 0, 1)
 

In [ ]:
# 6: Предзагрузка изображений в RAM

def preload_images(paths, desc="Загрузка"):
    """Загружает все изображения как numpy-массивы в shared memory (для DataLoader workers)."""
    imgs = []
    for p in tqdm(paths, desc=desc):
        imgs.append(load_rg(p))
    
    t = torch.from_numpy(np.stack(imgs))   # (N, H, W, 2)
    t.share_memory_()
    return t
 
 
if CACHE_IMAGES:
    print("Предзагрузка train изображений...")
    train_images_tensor = preload_images(train_paths, desc="  train")
    print("Предзагрузка val изображений...")
    val_images_tensor   = preload_images(val_paths,   desc="  val  ")
    print(f"RAM: {(train_images_tensor.nbytes + val_images_tensor.nbytes) / 1e9:.2f} GB")
else:
    train_images_tensor = None
    val_images_tensor   = None

In [ ]:
#  7: Dataset —  генерация патчей

class DemuxDataset(Dataset):
    """
    
    Каждый __getitem__ случайно выбирает изображение, конфигурацию K
    и вырезает случайный патч. Нет утечки данных между train и val.
 
    Для val  все индексы генерируются один раз при создании,
    
    """
    def __init__(
        self,
        images,          # torch.Tensor 
        configs,         
        n_samples,       # длина датасета 
        patch_size=PATCH_SIZE,
        augment=True,
        seed=None,       # если задан — фиксированные индексы (для val)
    ):
        self.images     = images
        self.configs    = configs
        self.n_samples  = n_samples
        self.patch_size = patch_size
        self.augment    = augment
 
        n_img = images.shape[0] if isinstance(images, torch.Tensor) else len(images)
        H = (images.shape[1] if isinstance(images, torch.Tensor) else 512)
        W = (images.shape[2] if isinstance(images, torch.Tensor) else 512)
 
        if seed is not None:
            rng = np.random.default_rng(seed)
            self._img_idx = rng.integers(0, n_img,                       n_samples)
            self._cfg_idx = rng.integers(0, len(configs),                n_samples)
            self._y0      = rng.integers(0, H - patch_size,              n_samples)
            self._x0      = rng.integers(0, W - patch_size,              n_samples)
        else:
            self._img_idx = None
 
    def _get_rg(self, img_idx):
        if isinstance(self.images, torch.Tensor):
            return self.images[img_idx].numpy().copy()
        return load_rg(self.images[img_idx])
 
    def __len__(self):
        return self.n_samples
 
    def __getitem__(self, idx):
        ps = self.patch_size
 
        if self._img_idx is not None:
            img_idx = int(self._img_idx[idx])
            cfg_idx = int(self._cfg_idx[idx])
            y0      = int(self._y0[idx])
            x0      = int(self._x0[idx])
        else:
            n_img   = (self.images.shape[0]
                       if isinstance(self.images, torch.Tensor)
                       else len(self.images))
            img_idx = np.random.randint(n_img)
            cfg_idx = np.random.randint(len(self.configs))
            H = (self.images.shape[1]
                 if isinstance(self.images, torch.Tensor) else 512)
            W = (self.images.shape[2]
                 if isinstance(self.images, torch.Tensor) else 512)
            y0 = np.random.randint(0, H - ps)
            x0 = np.random.randint(0, W - ps)
 
        rg     = self._get_rg(img_idx)          # (H, W, 2) float32
        config = self.configs[cfg_idx]
        H, W   = rg.shape[:2]
 
        K = make_mask(H, W, config)
        P = make_measurement(rg, K)
 
        # Вырезаем патч
        P_p  = P [y0:y0+ps, x0:x0+ps].copy()
        rg_p = rg[y0:y0+ps, x0:x0+ps].copy()
        K_p  = K [y0:y0+ps, x0:x0+ps].copy()
 
        if self.augment:
            # Геометрические аугментации (симметричны для P, rg, K)
            if np.random.rand() > 0.5:
                P_p  = np.fliplr(P_p).copy()
                rg_p = np.fliplr(rg_p).copy()
                K_p  = np.fliplr(K_p).copy()
            if np.random.rand() > 0.5:
                P_p  = np.flipud(P_p).copy()
                rg_p = np.flipud(rg_p).copy()
                K_p  = np.flipud(K_p).copy()
            if np.random.rand() > 0.5:
                # rot90 применяем только к P и rg; K тоже, иначе они не совпадут
                k90  = np.random.randint(1, 4)
                P_p  = np.rot90(P_p,  k90).copy()
                rg_p = np.rot90(rg_p, k90).copy()
                K_p  = np.rot90(K_p,  k90).copy()
 
            # Фотометрические аугментации
            gamma = np.random.uniform(0.8, 1.2)
            rg_p  = np.clip(rg_p ** gamma, 0, 1)
            P_p   = np.clip(P_p  ** gamma, 0, 1)
            shift = np.random.uniform(-0.05, 0.05)
            rg_p  = np.clip(rg_p + shift, 0, 1)
            P_p   = np.clip(P_p  + shift, 0, 1)
 
        cond   = config_to_cond(config)
        inp    = torch.tensor(np.stack([P_p, K_p], axis=0), dtype=torch.float32)
        target = torch.tensor(np.transpose(rg_p, (2, 0, 1)), dtype=torch.float32)
        cond_t = torch.tensor(cond, dtype=torch.float32)
        return inp, target, cond_t

In [ ]:
#  8: DataLoaderы

def worker_init_fn(worker_id):
    """Разные seed для каждого worker, иначе все генерируют одинаковые случайные числа."""
    np.random.seed(torch.initial_seed() % 2**32 + worker_id)
 
 
_images_train = train_images_tensor if CACHE_IMAGES else train_paths
_images_val   = val_images_tensor   if CACHE_IMAGES else val_paths
 
train_ds = DemuxDataset(
    _images_train, ALL_CONFIGS,
    n_samples=PATCHES_PER_EPOCH,
    augment=True, seed=None,
)
val_ds = DemuxDataset(
    _images_val, ALL_CONFIGS,
    n_samples=VAL_PATCHES,
    augment=False, seed=42,   # фиксированные патчи — val воспроизводим
)
 
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    worker_init_fn=worker_init_fn,
    prefetch_factor=4,        # каждый воркер готовит 4 батча вперёд
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    prefetch_factor=4,
)
print(f"Train батчей: {len(train_loader)} | Val батчей: {len(val_loader)}")

In [ ]:
# 9: Модель — UNet + ASPP-боттлнек + FiLM-кондиционирование

class ConvBlock(nn.Module):
    """Два свёрточных слоя с BatchNorm + ReLU."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)
 
 
class ASPP(nn.Module):
    """
    Atrous Spatial Pyramid Pooling (из DeepLab v3).
    Параллельные дилатированные свёртки с dilation=1,2,4,8 + глобальный пулинг.
    Рецептивное поле при patch=256: dilation=8 → 33×33 на боттлнеке → ~530 px исходника.
    Покрывает любой период маски.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid = out_ch // 4   # уменьшаем каждую ветку для эффективности
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_ch, mid, 3, padding=r, dilation=r, bias=False),
                nn.BatchNorm2d(mid),
                nn.ReLU(inplace=True),
            )
            for r in [1, 2, 4, 8]
        ])
        # Глобальный контекст через AdaptiveAvgPool
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_ch, mid, 1, bias=False),
            nn.ReLU(inplace=True),
        )
        # Проекция: 5 веток × mid → out_ch
        self.proj = nn.Sequential(
            nn.Conv2d(mid * 5, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),
        )
 
    def forward(self, x):
        h, w   = x.shape[2], x.shape[3]
        outs   = [b(x) for b in self.branches]
        gap_up = self.gap(x).expand(-1, -1, h, w)
        return self.proj(torch.cat(outs + [gap_up], dim=1))
 
 
class FiLM(nn.Module):
    """
    Feature-wise Linear Modulation: масштабирует и сдвигает карты признаков
    по внешнему вектору условия (тип и параметры маски K).
    Без FiLM сеть вынуждена угадывать паттерн K из числовых значений.
    """
    def __init__(self, num_features, cond_dim=COND_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_features * 2),   # gamma и beta
        )
 
    def forward(self, x, cond):
        # cond: (B, cond_dim)
        params        = self.net(cond)                         # (B, 2*C)
        gamma, beta   = params.chunk(2, dim=1)
        gamma         = gamma.unsqueeze(-1).unsqueeze(-1)      # (B, C, 1, 1)
        beta          = beta.unsqueeze(-1).unsqueeze(-1)
        return x * (1 + gamma) + beta                          # модуляция
 
 
class UNetASPP(nn.Module):
    """
    U-Net с четырьмя уровнями энкодера, ASPP-боттлнеком и FiLM-кондиционированием.
 
    Вход:  (B, 2, H, W)   — [P измерение, K маска]
    Условие: (B, 4)       — [тип маски, период, угол, фаза]
    
    """
    def __init__(self, in_ch=2, out_ch=2, cond_dim=COND_DIM):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
 
        # Энкодер
        self.enc1 = ConvBlock(in_ch,  32)
        self.enc2 = ConvBlock(32,     64)
        self.enc3 = ConvBlock(64,     128)
        self.enc4 = ConvBlock(128,    256)   # дополнительный уровень
 
        # ASPP боттлнек (параллельные dilation=1,2,4,8 + GAP)
        self.bot  = ASPP(256, 256)
 
        # FiLM — применяем к боттлнеку
        self.film = FiLM(256, cond_dim)
 
        # Декодер (skip connections — конкатенация)
        self.up4  = nn.ConvTranspose2d(256, 256, 2, stride=2)
        self.dec4 = ConvBlock(256 + 256, 128)
 
        self.up3  = nn.ConvTranspose2d(128, 128, 2, stride=2)
        self.dec3 = ConvBlock(128 + 128, 64)
 
        self.up2  = nn.ConvTranspose2d(64, 64, 2, stride=2)
        self.dec2 = ConvBlock(64 + 64, 32)
 
        self.up1  = nn.ConvTranspose2d(32, 32, 2, stride=2)
        self.dec1 = ConvBlock(32 + 32, 32)
 
        # 1×1 свёртка + sigmoid: вывод строго в [0,1]
        self.out  = nn.Sequential(
            nn.Conv2d(32, out_ch, 1),
            nn.Sigmoid(),
        )
 
    def forward(self, x, cond):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
 
        b  = self.bot(self.pool(e4))    # ASPP
        b  = self.film(b, cond)         # FiLM кондиционирование
 
        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
 
        return self.out(d1)
 
 
device = torch.device("cuda:1")
model  = UNetASPP(cond_dim=COND_DIM).to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model, device_ids=[1, 0], output_device=1)
    print(f"DataParallel: GPU 1 (главный) + GPU 0")
 
total_params = sum(p.numel() for p in model.parameters())
print(f"Устройство: {device}")
print(f"Параметров в модели: {total_params:,}")

In [ ]:
# Ячейка 10: Функция потерь — MSE + SSIM + Perceptual

class PerceptualLoss(nn.Module):
    """
    VGG16 perceptual loss на первых 16 слоях.
    Штрафует за структурные расхождения, которые MSE игнорирует.
    Предотвращает размытие при большом датасете.
    """
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features[:16].eval()
        for p in vgg.parameters():
            p.requires_grad = False
        self.vgg = vgg
        self.mse = nn.MSELoss()
 
    def forward(self, pred, target):
        # pred/target: (B, 2, H, W) → repeat до 3ch для VGG
        p3 = pred[:,  :1].repeat(1, 3, 1, 1)
        t3 = target[:, :1].repeat(1, 3, 1, 1)
        return self.mse(self.vgg(p3), self.vgg(t3))
 
 
class CombinedLoss(nn.Module):
    
    
    def __init__(self, alpha=0.6, beta=0.2, gamma=0.2):
        super().__init__()
        self.alpha   = alpha
        self.beta    = beta
        self.gamma   = gamma
        self.mse     = nn.MSELoss()
        self.percept = PerceptualLoss().to(device)
 
    def ssim_loss(self, pred, target):
        mu_p  = F.avg_pool2d(pred,        11, stride=1, padding=5)
        mu_t  = F.avg_pool2d(target,      11, stride=1, padding=5)
        sp    = F.avg_pool2d(pred**2,     11, stride=1, padding=5) - mu_p**2
        st    = F.avg_pool2d(target**2,   11, stride=1, padding=5) - mu_t**2
        spt   = F.avg_pool2d(pred*target, 11, stride=1, padding=5) - mu_p*mu_t
        C1, C2 = 0.01**2, 0.03**2
        ssim_map = ((2*mu_p*mu_t + C1) * (2*spt + C2)) / \
                   ((mu_p**2 + mu_t**2 + C1) * (sp + st + C2))
        return 1 - ssim_map.mean()
 
    def forward(self, pred, target):
        l_mse   = self.mse(pred, target)
        l_ssim  = self.ssim_loss(pred, target)
        l_perc  = self.percept(pred, target)
        return self.alpha * l_mse + self.beta * l_ssim + self.gamma * l_perc
 
 
criterion = CombinedLoss(alpha=0.6, beta=0.2, gamma=0.2)

In [ ]:
# 11: Оптимизатор и scheduler

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4,    
)
 
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=100,     # первый цикл 100 эпох
    T_mult=2,    # каждый следующий цикл вдвое длиннее
    eta_min=1e-6,
)

# периодические перезапуски lr помогают вырваться из локальных минимумов

In [ ]:
#  12: Функции чекпоинтов

def save_checkpoint(path, epoch, model, optimizer, scheduler,
                    best_val_loss, patience_cnt,
                    train_history, val_history):
    """
    Сохраняет полное состояние обучения.
    При любом сбое достаточно запустить обучение заново —
    оно продолжится с последней сохранённой эпохи.
    """
    torch.save({
        "epoch":           epoch,
        "model_state":     (model.module.state_dict()
                if isinstance(model, nn.DataParallel)
                else model.state_dict()),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "best_val_loss":   best_val_loss,
        "patience_cnt":    patience_cnt,
        "train_history":   train_history,
        "val_history":     val_history,
    }, path)
    print(f"  [ckpt] сохранён: {path}  (epoch {epoch})")
 
 
def load_checkpoint(path, model, optimizer, scheduler, scaler=None):
    ckpt = torch.load(path, map_location=device, weights_only=False)  # ← исправлено (см. ошибку 2)

    state = ckpt["model_state"]
    target = model.module if isinstance(model, nn.DataParallel) else model

    first_key = next(iter(state.keys()))
    if not first_key.startswith("module."):
        target.load_state_dict(state)
    else:
        model.load_state_dict(state)

    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])

    # ← scaler восстанавливается здесь, второй torch.load() больше не нужен
    if scaler is not None and "scaler_state" in ckpt:
        scaler.load_state_dict(ckpt["scaler_state"])
        print(f"  [ckpt] scaler восстановлен")

    print(f"  [ckpt] восстановлен: {path}  (epoch {ckpt['epoch']})")
    return (
        ckpt["epoch"] + 1,
        ckpt["best_val_loss"],
        ckpt["patience_cnt"],
        ckpt["train_history"],
        ckpt["val_history"],
    )

In [ ]:
# 13: Обучение с автосохранением и возобновлением

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

start_epoch   = 0
best_val_loss = float("inf")
patience_cnt  = 0
train_history = []
val_history   = []

# AMP scaler — float16 на forward, float32 на weights
scaler = torch.amp.GradScaler('cuda')

if os.path.exists(CHECKPOINT_PATH):
    print(f"Найден чекпоинт {CHECKPOINT_PATH} — возобновляем обучение...")
    # Один load — передаём scaler напрямую в load_checkpoint
    start_epoch, best_val_loss, patience_cnt, train_history, val_history = \
        load_checkpoint(CHECKPOINT_PATH, model, optimizer, scheduler, scaler)
else:
    print("Чекпоинт не найден — обучение с нуля.")

# Основной цикл
for epoch in range(start_epoch, EPOCHS):

    # Train 
    model.train()
    train_loss = 0.0
    loop = tqdm(train_loader, desc=f"Epoch {epoch:03d} [train]", leave=False)
    for inp, target, cond in loop:
        inp, target, cond = inp.to(device), target.to(device), cond.to(device)
        optimizer.zero_grad(set_to_none=True)

        # AMP autocast — вычисления в float16, экономит память и ускоряет
        with torch.amp.autocast('cuda'):
            pred = model(inp, cond)
            loss = criterion(pred, target)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        loop.set_postfix(loss=f"{loss.item():.4f}")

    # Validation 
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inp, target, cond in val_loader:
            inp, target, cond = inp.to(device), target.to(device), cond.to(device)
            with torch.amp.autocast('cuda'):
                val_loss += criterion(model(inp, cond), target).item()

    train_loss /= len(train_loader)
    val_loss   /= len(val_loader)
    train_history.append(train_loss)
    val_history.append(val_loss)
    scheduler.step()

    lr_now = optimizer.param_groups[0]["lr"]

    # Early stopping + сохранение лучшей модели 
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_cnt  = 0
        # Сохраняем только веса модели (без DataParallel обёртки)
        state = (model.module if isinstance(model, nn.DataParallel) else model).state_dict()
        torch.save(state, BEST_MODEL_PATH)
        print(f"Epoch {epoch:03d}  train={train_loss:.4f}  val={val_loss:.4f}"
              f"  lr={lr_now:.6f}  ★ новый лучший")
    else:
        patience_cnt += 1
        if epoch % 25 == 0:
            print(f"Epoch {epoch:03d}  train={train_loss:.4f}  val={val_loss:.4f}"
                  f"  lr={lr_now:.6f}  patience={patience_cnt}/{PATIENCE}")
        if patience_cnt >= PATIENCE:
            print(f"\nEarly stopping на epoch {epoch}")
            # Сохранить финальный чекпоинт перед остановкой
            torch.save({
                "epoch":           epoch,
                "model_state":     (model.module if isinstance(model, nn.DataParallel)
                                    else model).state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "scaler_state":    scaler.state_dict(),
                "best_val_loss":   best_val_loss,
                "patience_cnt":    patience_cnt,
                "train_history":   train_history,
                "val_history":     val_history,
            }, CHECKPOINT_PATH)
            break

    #  Периодическое сохранение чекпоинта 
    if epoch % CHECKPOINT_EVERY == 0:
        torch.save({
            "epoch":           epoch,
            "model_state":     (model.module if isinstance(model, nn.DataParallel)
                                else model).state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state":    scaler.state_dict(),   # ← scaler сохраняется
            "best_val_loss":   best_val_loss,
            "patience_cnt":    patience_cnt,
            "train_history":   train_history,
            "val_history":     val_history,
        }, CHECKPOINT_PATH)
        print(f"  [ckpt] сохранён: {CHECKPOINT_PATH}  (epoch {epoch})")

#  Загружаем лучшие веса 
state = torch.load(BEST_MODEL_PATH, map_location=device)
(model.module if isinstance(model, nn.DataParallel) else model).load_state_dict(state)
print(f"\nЛучший val_loss: {best_val_loss:.4f}")

#  График потерь 
plt.figure(figsize=(10, 4))
plt.plot(train_history, label="train")
plt.plot(val_history,   label="val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Val loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=120)
plt.show()

In [ ]:
import torch
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}  |  compute {props.major}.{props.minor}  |  {props.total_memory/1e9:.1f} GB")

print(f"\nPyTorch версия: {torch.__version__}")
print(f"CUDA версия:    {torch.version.cuda}")

In [ ]:
import subprocess
subprocess.run([
    "pip", "install", "--upgrade",
    "torch==2.1.2+cu118",
    "torchvision==0.16.2+cu118",
    "--index-url", "https://download.pytorch.org/whl/cu118"
], check=True)

In [ ]:
import torch
print(torch.__version__)          # 2.1.2+cu118
print(torch.cuda.is_available())  # True
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")